In [ ]:
%pip install "numpy<2" #for compatibility issues
%pip install wandb

In [ ]:
import wandb
wandb.login()
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#loading
from keras.datasets import fashion_mnist as mnist
(X_train, Y_train), (X_test, Y_test)=mnist.load_data()

In [ ]:
#EDA
print(f"TRAINING SET (feature|label): {np.shape(X_train)}"+f" | {np.shape(Y_train)}")
print(f"TEST SET (target|label): {np.shape(X_test)}"+f" | {np.shape(Y_test)}")

labels, indices, counts = np.unique(Y_train, return_counts=True, return_index=True)
print(f"labels: {labels}")
print(f"counts: {counts}")

#image from every category
fig, axes = plt.subplots(2, 5)
axes=axes.flatten()
for ax, index in zip(axes, indices):
    ax.imshow(X_train[index])
    fig.savefig("image_per_class.png", dpi=300, bbox_inches='tight')

In [ ]:
#pre-processing
#flatten images
X_flat = X_train.reshape(X_train.shape[0], -1)/255.0 
X_test_flat = X_test.reshape(X_test.shape[0], -1)/255.0

#shuffling the training-set
perm=np.random.permutation(60000) 
X_train_shuffled=X_flat[perm]
Y_train_shuffled=Y_train[perm]

#training-valdiation split [10% split]
X_val, X_training = X_train_shuffled[:6000], X_train_shuffled[6000:]
Y_val, Y_training = Y_train_shuffled[:6000], Y_train_shuffled[6000:]

In [ ]:
#Feedforward Network
#creating w and b matrices for every layer and initialising them with support for xavier and gaussian
def build_params(sizes, init):
    params=[]
    for i in range(len(sizes)-1):
        if init.lower() == 'random':
            w=np.random.randn(sizes[i], sizes[i+1])*0.01
        elif init.lower() == 'xavier':
            L = np.sqrt(6/(sizes[i]+sizes[i+1]))
            w=np.random.uniform(-L, L, (sizes[i], sizes[i+1]))
        b=np.zeros(sizes[i+1])
        params.append({'w':w, 'b':b})
    return params

In [ ]:
#forward pass
def softmax(x):
    x_norm=x-np.max(x, axis=1, keepdims=True)
    return np.exp(x_norm)/np.exp(x_norm).sum(axis=1, keepdims=True)

def forward_pass(data, params, a_fn):
    a=[data]  #activation at every layer and final output
    z=[] #pre-activation at every layer
    for i in range(len(params)):
        z.append(a[i]@params[i]['w']+params[i]['b'])
        if i==len(params)-1:
            a.append(softmax(z[i]))
        else:
            if a_fn.lower() == 'sigmoid':
                a.append(1/(1+np.exp(-z[i])))
            elif a_fn.lower() == 'tanh':
                a.append(np.tanh(z[i]))
            elif a_fn.lower() == 'relu':
                a.append(np.maximum(0, z[i]))
    return a, z

def loss(output, labels, fn):
    one_hot = np.eye(10)[labels] #one-hot encoding
    N = output.shape[0] #batch-size (used for calculating the average/batch loss)
    if fn.lower() == 'cross_entropy':
        true_class_probs = (output*one_hot).sum(axis=1)
        loss = np.mean(-np.log(true_class_probs + 1e-9))
        grad = (output-one_hot)/N
        return loss, grad
    elif fn.lower() == 'mse':
        value = np.mean(np.sum((output - one_hot)**2, axis=1))
        diff = output - one_hot
        s = np.sum(diff * output, axis=1, keepdims=True)
        grad = (2.0 / N) * output * (diff - s)
        return value, grad
    

In [ ]:
#backprop
#dx represents gradient of L w.r.t x
def backprop(output_gradient, params, a, z, a_fn):
    grads = [None]*len(params)
    dz = output_gradient
    for i in reversed(range(len(params))):
        dw = a[i].T@ dz
        db = dz.sum(axis=0)
        grads[i] = {'dw': dw, 'db': db}
        if i>0:
            dz = (dz@params[i]['w'].T)*activation_derivative(a[i], a_fn)
    return grads 

def activation_derivative(a, a_fn):
    if a_fn.lower() == 'sigmoid':
        return a*(1-a)
    elif a_fn.lower() == 'tanh':
        return (1-(a**2))
    elif a_fn.lower() == 'relu':
        return np.where(a > 0, 1.0, 0.0)

In [ ]:
#optimisers
class SGD:
    def __init__(self, lr, weight_decay=0.0):
        self.lr = lr
        self.wd = weight_decay
    def step(self, params, grads):
        for i in range(len(params)):
            params[i]['w'] -= self.lr*(grads[i]['dw'] + self.wd*params[i]['w'])
            params[i]['b'] -= self.lr*grads[i]['db']

class Momentum:
    def __init__(self, lr, m, params, weight_decay=0.0):
        self.lr = lr
        self.m = m
        self.wd = weight_decay
        self.velocity = [{'w': np.zeros_like(p['w']), 'b': np.zeros_like(p['b'])} for p in params]
    def step(self, params, grads):
        for i in range(len(params)):
            dw = grads[i]['dw'] + self.wd * params[i]['w']
            self.velocity[i]['w'] = self.m*self.velocity[i]['w']+grads[i]['dw'] + dw
            self.velocity[i]['b'] = self.m*self.velocity[i]['b']+grads[i]['db']
            params[i]['w'] -= self.lr*self.velocity[i]['w']
            params[i]['b'] -= self.lr*self.velocity[i]['b']

class Nesterov:
    def __init__(self, lr, m, params, weight_decay=0.0):
        self.lr = lr
        self.m = m
        self.wd = weight_decay
        self.velocity = [{'w': np.zeros_like(p['w']), 'b': np.zeros_like(p['b'])} for p in params]
    def step(self, params, grads):
        for i in range(len(params)):
            dw = grads[i]['dw'] + self.wd * params[i]['w']
            self.velocity[i]['w'] = self.m*self.velocity[i]['w']+grads[i]['dw'] + dw
            self.velocity[i]['b'] = self.m*self.velocity[i]['b']+grads[i]['db']
            params[i]['w'] -= self.lr*(self.m*self.velocity[i]['w']+grads[i]['dw'])
            params[i]['b'] -= self.lr*(self.m*self.velocity[i]['b']+grads[i]['db'])

class RMSProp:
    def __init__(self, lr, beta, params, eps=1e-8, weight_decay=0.0):
        self.lr = lr
        self.beta = beta
        self.eps = eps
        self.wd = weight_decay
        self.v = [{'w': np.zeros_like(p['w']),
                   'b': np.zeros_like(p['b'])} for p in params]
    def step(self, params, grads):
        for i in range(len(params)):
            dw = grads[i]['dw'] + self.wd * params[i]['w']
            self.v[i]['w'] = self.beta * self.v[i]['w'] + (1 - self.beta) * grads[i]['dw']*dw**2
            self.v[i]['b'] = self.beta * self.v[i]['b'] + (1 - self.beta) * grads[i]['db']**2
            params[i]['w'] -= self.lr * grads[i]['dw'] / (np.sqrt(self.v[i]['w']) + self.eps)
            params[i]['b'] -= self.lr * grads[i]['db'] / (np.sqrt(self.v[i]['b']) + self.eps)

class Adam:
    def __init__(self, lr, beta1, beta2, params, eps=1e-8, weight_decay=0.0):
        self.lr = lr
        self.beta1 = beta1            
        self.beta2 = beta2            
        self.eps = eps
        self.t = 0
        self.wd = weight_decay
        self.m = [{'w': np.zeros_like(p['w']),
                   'b': np.zeros_like(p['b'])} for p in params]
        self.v = [{'w': np.zeros_like(p['w']),
                   'b': np.zeros_like(p['b'])} for p in params]
    def step(self, params, grads):
        self.t += 1
        for i in range(len(params)):
            for key, gkey in [('w', 'dw'), ('b', 'db')]:
                g = grads[i][gkey]
                if key == 'w':
                    g = g + self.wd * params[i]['w']
                self.m[i][key] = self.beta1 * self.m[i][key] + (1 - self.beta1) * g
                self.v[i][key] = self.beta2 * self.v[i][key] + (1 - self.beta2) * g**2
                m_hat = self.m[i][key] / (1 - self.beta1**self.t)
                v_hat = self.v[i][key] / (1 - self.beta2**self.t)
                params[i][key] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

class Nadam:
    def __init__(self, lr, beta1, beta2, params, eps=1e-8, weight_decay=0.0):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        self.wd = weight_decay
        self.m = [{'w': np.zeros_like(p['w']),
                   'b': np.zeros_like(p['b'])} for p in params]
        self.v = [{'w': np.zeros_like(p['w']),
                   'b': np.zeros_like(p['b'])} for p in params]
    def step(self, params, grads):
        self.t += 1
        for i in range(len(params)):
            for key, gkey in [('w', 'dw'), ('b', 'db')]:
                g = grads[i][gkey]
                if key == 'w':
                    g = g + self.wd * params[i]['w']
                self.m[i][key] = self.beta1 * self.m[i][key] + (1 - self.beta1) * g
                self.v[i][key] = self.beta2 * self.v[i][key] + (1 - self.beta2) * g**2
                m_hat = self.m[i][key] / (1 - self.beta1**self.t)
                v_hat = self.v[i][key] / (1 - self.beta2**self.t)
                m_nesterov = self.beta1 * m_hat + (1 - self.beta1) * g / (1 - self.beta1**self.t)
                params[i][key] -= self.lr * m_nesterov / (np.sqrt(v_hat) + self.eps)

In [ ]:
#training loop
def train(params, X_tr, Y_tr, optimizer, a_fn, loss_fn, epochs, batch_size):
    N = len(X_tr)
    for epoch in range(epochs):
        perm = np.random.permutation(N)
        X_sh = X_tr[perm]
        Y_sh = Y_tr[perm]

        epoch_loss = 0.0
        n_batches = 0
        for start in range(0, N, batch_size):
            bX = X_sh[start:start + batch_size]
            bY = Y_sh[start:start + batch_size]

            a, z = forward_pass(bX, params, a_fn)
            loss_val, output_grad = loss(a[-1], bY, loss_fn)
            grads = backprop(output_grad, params, a, z, a_fn)
            optimizer.step(params, grads)

            epoch_loss += loss_val
            n_batches += 1
        print(f"epoch {epoch+1:2d}   avg loss {epoch_loss / n_batches:.4f}")
    return params

In [ ]:
#config for hyperparameter sweep
sweep_config = {
    'method': 'random',
    'metric': {'name': 'val_accuracy', 'goal': 'maximize'},
    'parameters': {
        'epochs':        {'values': [5, 10]},
        'num_layers':    {'values': [3, 4, 5]},
        'hidden_size':   {'values': [32, 64, 128]},
        'weight_decay':  {'values': [0, 0.0005, 0.5]},
        'learning_rate': {'values': [1e-3, 1e-4]},
        'optimizer':     {'values': ['sgd','momentum','nesterov','rmsprop','adam','nadam']},
        'batch_size':    {'values': [16, 32, 64]},
        'weight_init':   {'values': ['random', 'Xavier']},
        'activation':    {'values': ['sigmoid', 'tanh', 'relu']},
        'loss':          {'values': ['cross_entropy']},
    }
}

In [ ]:
#accuracy metrics
def accuracy(params, X, Y, a_fn):
    a, _ = forward_pass(X, params, a_fn)
    preds = np.argmax(a[-1], axis=1)     
    return np.mean(preds == Y)            

#helper for choosing optimisers
def make_optimizer(name, params, lr, m=0.9, beta=0.9,
                   beta1=0.9, beta2=0.999, wd=0.0):
    name = name.lower()
    if name == 'sgd':      return SGD(lr, weight_decay=wd)
    if name == 'momentum': return Momentum(lr, m, params, weight_decay=wd)
    if name == 'nesterov': return Nesterov(lr, m, params, weight_decay=wd)
    if name == 'rmsprop':  return RMSProp(lr, beta, params, weight_decay=wd)
    if name == 'adam':     return Adam(lr, beta1, beta2, params, weight_decay=wd)
    if name == 'nadam':    return Nadam(lr, beta1, beta2, params, weight_decay=wd)
    raise ValueError(f"unknown optimizer: {name}")

In [ ]:
def sweep_run():
    wandb.init()
    config = wandb.config

    sizes = [784] + [config.hidden_size] * config.num_layers + [10]
    params = build_params(sizes, config.weight_init)

    opt = make_optimizer(config.optimizer, params, config.learning_rate, config.weight_decay)

    for epoch in range(config.epochs):    
        train(params, X_train, Y_train, opt, 
              a_fn=config.activation, 
              loss_fn=config.loss, 
              epochs=1, 
              batch_size=config.batch_size)
        
        #train metrics
        va_tr, _ = forward_pass(X_train, params, config.activation)
        train_loss, _ = loss(va_tr[-1], Y_train, config.loss)
        train_acc = accuracy(params, X_train, Y_train, config.activation)
        
        #validation metrics
        va_val, _ = forward_pass(X_val, params, config.activation)
        val_loss, _ = loss(va_val[-1], Y_val, config.loss)
        val_acc = accuracy(params, X_val, Y_val, config.activation)

        #logging
        wandb.log({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'train_accuracy': train_acc,
            'val_accuracy': val_acc
        })

In [ ]:
#hyperparam sweep
'''
sweep_id = wandb.sweep(sweep_config, project='fashion-mnist-mlp')
print("sweep_id:", sweep_id)
wandb.agent(sweep_id, function=sweep_run, count=50)
'''

In [ ]:
#MNIST-Digits test
from keras.datasets import mnist
(X_train_raw_d, Y_train_d), (X_test_raw_d, Y_test_d) = mnist.load_data()


X_train_digits = X_train_raw_d.reshape(X_train_raw_d.shape[0], -1) / 255.0
X_test_digits = X_test_raw_d.reshape(X_test_raw_d.shape[0], -1) / 255.0

#config 1 (best config model)
np.random.seed(0)
sizes = [784] + [64]*3 + [10]                    
params_digits = build_params(sizes, 'random')            
opt_digits = make_optimizer('nadam', params_digits, 1e-3, wd=0.0)

train(params_digits, X_train_digits, Y_train_d, opt_digits,
      a_fn='tanh', loss_fn='cross_entropy',
      epochs=10, batch_size=32)                    

test_acc_1 = accuracy(params_digits, X_test_digits, Y_test_d, 'tanh')
print(f"Test accuracy: {test_acc_1:.4f}")

In [ ]:
#config 2 (shallower model)
np.random.seed(0)
sizes_2 = [784, 64, 10]                    
params_2 = build_params(sizes_2, 'random')            
opt_2 = make_optimizer('nadam', params_2, 1e-3, wd=0.0)

train(params_2, X_train_digits, Y_train_d, opt_2,
      a_fn='tanh', loss_fn='cross_entropy',
      epochs=10, batch_size=32)                    

test_acc_2 = accuracy(params_2, X_test_digits, Y_test_d, 'tanh')

print(f"Test accuracy: {test_acc_2:.4f}")

In [ ]:
#config 3 (same as #2 but with sgd as optimiser and lr of 1e-2, i.e. 10 times as fast)
np.random.seed(0)
sizes_3 = [784, 64, 10]                    
params_3 = build_params(sizes_3, 'random')            
opt_3 = make_optimizer('sgd', params_3, 1e-2, wd=0.0)

train(params_3, X_train_digits, Y_train_d, opt_3,
      a_fn='tanh', loss_fn='cross_entropy',
      epochs=10, batch_size=32)                    

test_acc_3 = accuracy(params_3, X_test_digits, Y_test_d, 'tanh')

print(f"Test accuracy: {test_acc_3:.4f}")